# Phase 3 Extension Walkthrough

This notebook demonstrates the interaction-aware InstaSHAP extension on the sklearn `friedman1` benchmark.

## 1. Imports

In [ ]:
from phase2.data.data_loader import load_dataset
from phase2.models.base_model import predict_black_box, train_black_box_model
from phase2.explainers.exact_shap import compute_exact_shap
from phase2.explainers.instashap_explainer import InstaSHAPExplainer
from phase2.utils import compute_alignment_metrics, configure_plotting, seed_everything, select_background_frame
from phase3.extension.interaction_aware_surrogate import train_interaction_aware_surrogate
from phase3.extension.enhanced_instashap import compute_interaction_aware_instashap

seed_everything()
configure_plotting()

## 2. Load the Interaction-Heavy Dataset

In [ ]:
bundle = load_dataset('friedman1')
bundle.X_train.head()

## 3. Fit the Black-Box Model and Exact SHAP Baseline

In [ ]:
model_bundle = train_black_box_model(
    X_train=bundle.X_train,
    y_train=bundle.y_train,
    task=bundle.task,
    model_name='xgboost',
)
background = select_background_frame(bundle.X_train, max_rows=75)
X_explain = bundle.X_test.head(64)
exact = compute_exact_shap(
    model=model_bundle.model,
    X_background=background,
    X_explain=X_explain,
    task=bundle.task,
    feature_names=bundle.feature_names,
)

## 4. Compare Original InstaSHAP with the Interaction-Aware Extension

In [ ]:
original = InstaSHAPExplainer(
    black_box_model=model_bundle.model,
    task=bundle.task,
    feature_names=bundle.feature_names,
).fit(bundle.X_train).explain(X_explain)

interaction_surrogate = train_interaction_aware_surrogate(
    X_train=bundle.X_train,
    black_box_predictions=predict_black_box(model_bundle.model, bundle.X_train, task=bundle.task),
    feature_names=bundle.feature_names,
    interaction_pairs=[('x_1', 'x_2')],
)
enhanced = compute_interaction_aware_instashap(
    surrogate=interaction_surrogate.surrogate,
    X=X_explain,
    reference_data=bundle.X_train,
    feature_names=bundle.feature_names,
)

compute_alignment_metrics(exact.values, original.values), compute_alignment_metrics(exact.values, enhanced.values)

## 5. Run the Full Phase 3 Scripts

```bash
python -m phase3.experiments.experiment_gap_demonstration
python -m phase3.experiments.experiment_extension_accuracy
python -m phase3.experiments.experiment_extension_runtime
python -m phase3.experiments.experiment_comparison
```